# Land Surveying

Land surveying has traditionally been based, in essence, on the
application of trigonometric functions. This is why it is also referred
to as triangulation. 


::: {.callout-note title="Trigonometric Fundamentals"}

In a general triangle (cf. @fig-dreieck), the following two theorems
hold:

![General triangle.](dreieck.svg){#fig-dreieck}

**Sine rule**

$$
\frac{a}{\sin \alpha} = \frac{b}{\sin \beta} = \frac{c}{\sin \gamma}
$$

**Cosine rule**

\begin{align*}
a^2 &= b^2 + c^2 - 2 \cdot b \cdot c \cdot \cos \alpha \\
b^2 &= c^2 + a^2 - 2 \cdot c \cdot a \cdot \cos \beta \\
c^2 &= a^2 + b^2 - 2 \cdot a \cdot b \cdot \cos \gamma
\end{align*}

:::


## Practical Exercise

In what follows, this will be illustrated and applied by surveying a
distance across inaccessible terrain.

Consider the following initial situation (cf. @fig-vermessung):

![Survey sketch[@doberkatEbeneTrigonometrieAnalytische2024, p. 56].](vermessung.svg){#fig-vermessung}

We are looking for the distance $\overline{AB}$ ($\ell$). What can be
measured are the distance $\overline{CD}$ ($s$) as well as the angles
$\alpha_1$, $\alpha_2$, $\beta_1$ and $\beta_2$. Here, $\alpha_1$ and
$\alpha_2$ are located at point $D$, while $\beta_1$ and $\beta_2$ are
located at point $C$.

### Working Steps 

The idea is to link the baseline $s = \overline{CD}$ to the distance we
are looking for by means of two auxiliary triangles. From the triangle
$C\,D\,B$ we obtain the distance $\overline{CB}$, and from the triangle
$C\,D\,A$ the distance $\overline{CA}$. Both distances share the point
$C$; the angle enclosed between them is $\beta_2 - \beta_1$. The target
distance can then be determined in the triangle $C\,A\,B$ using the
cosine rule.

With the measured quantities, the following missing quantities can be
calculated in turn:

1. $\gamma_1 = 180^{\circ} - (\alpha_1 + \beta_1)$
2. Using the sine rule (the side opposite $\alpha_1$, i.e.
   $\overline{CB}$): $a = \dfrac{s \cdot \sin \alpha_1}{\sin \gamma_1}$
3. $\gamma_2 = 180^{\circ} - (\alpha_2 + \beta_2)$
4. Using the sine rule (the side opposite $\alpha_2$, i.e.
   $\overline{CA}$): $b = \dfrac{s \cdot \sin \alpha_2}{\sin \gamma_2}$
5. Using the cosine rule in the triangle $C\,A\,B$:
   $\ell =\sqrt{a^2 + b^2 - 2ab \cos (\beta_2 - \beta_1)}$


### Calculations Using Python

It is considerably easier to pass all measured values to a Python
function than to calculate every step manually, one at a time, with a
pocket calculator. Note, however, that Python performs angle
calculations in radians rather than in degrees.

::: {.callout-note}

## Angles in Radian Measure

:::: {.columns}

::: {.column width="50%"}

![](circle.svg)
:::

::: {.column width="50%"}
The radian measure $\widehat{\varphi}$ of an angle is the ratio of the
length of the circular arc cut out by the angle $\varphi$ to the radius
of the circle. It is also denoted by $\operatorname{arc} \varphi$.

$$\widehat{\varphi} = \frac{b}{r} = \operatorname{arc} \varphi =
\frac{\pi}{180^\circ} \varphi \iff \varphi = \frac{180^\circ}{\pi}
\widehat{\varphi}$$ 

On the unit circle ($r=1$): $\widehat{\varphi} = b$. The unit of
radian measure is the radian (rad). An angle of 1 rad corresponds to a
circular arc that has the same length as the
radius.[@durandiFormelnTabellenBegriffe2022, p. 91]  
:::
:::
:::

Fortunately, Python provides the function `math.radians()` in the
`math` module, which converts degrees into radians. To use it, the
module must first be imported with `import math`. The `math` module
also provides all the trigonometric functions required (`math.sin()`
and `math.cos()`).

To keep the Python function as lean and readable as possible, we first
implement a helper function that converts degrees into radians. Since
there are also compasses calibrated in mils (A‰) and gon, the function
should alternatively be able to convert values in these units as well.



In [3]:
import math

In [4]:
def angle_converter(angle: float, mil: bool = False, gon: bool = False) -> float:
    """Converts an angle into radian measure (radians).

    Supports conversion from degrees (default), artillery mils or gon.
    Only one mode may be active at a time.

    Args:
        angle (float): The angle value to be converted.
        mil (bool, optional): If True, the angle is interpreted as artillery
            mils (A‰). Defaults to False.
        gon (bool, optional): If True, the angle is interpreted as gon
            (gradians). Defaults to False.

    Returns:
        float: The converted angle in radians.

    Raises:
        ValueError: If both 'mil' and 'gon' are set to True.
    """
    if mil and gon:
        raise ValueError(
            "Only 'mil=True' (artillery) OR 'gon=True' may be selected."
        )

    if mil:
        return angle * (math.pi / 3200)
    elif gon:
        return angle * (math.pi / 200)
    else:
        return math.radians(angle)

How the angles required for the calculation are measured shall be shown
using the example of $\alpha_1$ in @fig-vermessung. To do so, the
azimuth towards $B$ and towards $C$ is measured with the compass from
position $D$. The difference between the two measurements corresponds
to $\alpha_1$.

To implement this calculation in a Python function, it must be
described as an algorithm. In doing so, the case must be considered in
which the two measurements enclose the direction of north. To cover
this situation as well, we make use of *modular subtraction*. 

::: {.callout-note}

## Modular Subtraction

Modular arithmetic is about transferring the familiar arithmetic with
numbers from an infinite set to finite sets of
numbers[@iwanowskiDiskreteMathematikMit2021, p. 166]. This ensures
that the result of the angle calculation remains within the compass
rose.

**In general, for 360°** $(a - b) \pmod{360}$

1. Calculate the difference: $d = a - b$
2. Take the modulo: $r = d \pmod{360}$

The third step that is customary in the classical presentation – "if
the result is negative, add $360$ once" – is unnecessary in Python:
with a positive divisor, the `%` operator always returns a result in
the range $[0, 360)$ and thus normalises automatically.

**Numerical example** $40^{\circ} - 110^{\circ}$

1. $40 - 110 = - 70$
2. $-70 < 0: \text{true}$
3. $−70 + 360 = 290$
4. $40 − 110 \equiv 290 \pmod{360}$

:::

Implemented as a Python function, this looks as follows:


In [2]:
def angle_diff(angle1: float, angle2: float) -> float:
    """Calculates the modular difference (angle1 - angle2) modulo 360 degrees.
    
    Automatically accounts for the crossing of north by mapping negative
    differences into the positive cycle [0, 360).

    Args:
        angle1 (float): The initial angle (e.g. first bearing).
        angle2 (float): The angle to be subtracted (e.g. second bearing).

    Returns:
        float: The modular angle difference in the range from 0 to <360 degrees.
    """
    return (angle1 - angle2) % 360

We now also need a helper function for the sine rule.


In [ ]:
def sine_rule(length: float, opposite_angle: float, ref_angle: float) -> float:
    """Calculates a side of a triangle using the sine rule.

    Determines the side opposite the angle ``opposite_angle``, starting
    from a known side ``length`` and the angle ``ref_angle`` opposite
    that side:

        required side = length * sin(opposite_angle) / sin(ref_angle)

    All angles must be passed in radian measure (radians).

    Args:
        length (float): Known side (here the baseline s).
        opposite_angle (float): Angle opposite the side we are looking for
            (in radians).
        ref_angle (float): Angle opposite the known side ``length``
            (in radians).

    Returns:
        float: The length of the side we are looking for.
    """
    return (length * math.sin(opposite_angle)) / math.sin(ref_angle)

The required helper functions are now in place, and the actual
calculation along the formula $\ell =\sqrt{a^2 + b^2 - 2ab
\cos (\beta_2 - \beta_1)}$ can be implemented as a Python
function. 


In [ ]:
def triangulation(s: float,
                  alpha1_a: float, alpha1_b: float,
                  alpha2_a: float, alpha2_b: float,
                  beta1_a: float, beta1_b: float,
                  beta2_a: float, beta2_b: float,
                  mil: bool = False,
                  gon: bool = False) -> float:
    """Calculates an inaccessible distance by triangulation from compass bearings.

    Determines the length of the unknown target distance AB (l) from a known
    base length CD (s) and the corresponding azimuth bearings. From each pair
    of bearings, an interior angle of a triangle is formed via the modular
    difference. Using the sine rule, the distances from C to B and from C to A
    are then determined; finally, the cosine rule in the triangle C-A-B yields
    the target distance, where the angle enclosed by the two distances at C
    corresponds to the difference beta2 - beta1.

    All four partial angles are reconstructed from two compass bearings
    each (a and b).

    Args:
        s (float): The measured, accessible baseline (CD).
        alpha1_a (float): First compass bearing for the partial angle alpha 1 (at D).
        alpha1_b (float): Second compass bearing for the partial angle alpha 1 (at D).
        alpha2_a (float): First compass bearing for the partial angle alpha 2 (at D).
        alpha2_b (float): Second compass bearing for the partial angle alpha 2 (at D).
        beta1_a (float): First compass bearing for the partial angle beta 1 (at C).
        beta1_b (float): Second compass bearing for the partial angle beta 1 (at C).
        beta2_a (float): First compass bearing for the partial angle beta 2 (at C).
        beta2_b (float): Second compass bearing for the partial angle beta 2 (at C).
        mil (bool, optional): If True, all bearing values are interpreted as
            artillery mils (A‰). Defaults to False.
        gon (bool, optional): If True, all bearing values are interpreted as
            gon (gradians). Defaults to False.

    Returns:
        float: The calculated length of the inaccessible target distance (l).

    Raises:
        ValueError: If, via the helper functions, both 'mil' and 'gon' are
            set to True.
    """
    # Form the interior partial angle from each pair of bearings as a modular difference.
    input_angles = [alpha1_a, alpha1_b, alpha2_a, alpha2_b,
                    beta1_a, beta1_b, beta2_a, beta2_b]
    angles = []
    for i in range(0, len(input_angles), 2):
        diff = angle_diff(input_angles[i], input_angles[i + 1])
        angles.append(diff)

    # Convert into radians.
    alpha1, alpha2, beta1, beta2 = (
        angle_converter(angle, mil, gon) for angle in angles
    )

    # Auxiliary triangles on the baseline s: gamma lies opposite s in each case.
    gamma1 = math.radians(180) - (alpha1 + beta1)
    gamma2 = math.radians(180) - (alpha2 + beta2)

    # The sine rule yields the side opposite alpha (from C): CB and CA respectively.
    side_cb = sine_rule(s, alpha1, gamma1)
    side_ca = sine_rule(s, alpha2, gamma2)

    # Cosine rule in the triangle C-A-B; the enclosed angle at C is beta2 - beta1.
    l = math.sqrt(
        side_cb**2 + side_ca**2
        - 2 * side_cb * side_ca * math.cos(beta2 - beta1)
    )
    return l